# Nginx como Proxy Inverso

Enrutar tráfico con nginx hacia tus contenedores

## Introducción

Nginx es el servidor web más usado en producción. Como reverse proxy, actúa de intermediario entre internet y tus contenedores Docker: recibe las peticiones HTTP/HTTPS, las redirige al contenedor correcto, y devuelve las respuestas. En esta lección aprenderás a configurarlo desde cero.

### Objetivos de Aprendizaje

- Entender la estructura de nginx.conf (events, http, server, location)
- Configurar un reverse proxy básico hacia un contenedor Docker
- Servir múltiples aplicaciones en el mismo servidor con diferentes dominios
- Servir archivos estáticos (build de React/Vite)
- Activar compresión gzip y headers de caché
- Implementar rate limiting para proteger tu API

## Estructura de nginx.conf

> El archivo nginx.conf tiene una estructura jerárquica de bloques. El bloque http contiene toda la configuración web. Dentro van los bloques server (uno por dominio/puerto) y dentro de cada server van los bloques location que definen cómo manejar cada ruta.

In [ ]:
estructura = """
# Bloque global - configuración del proceso nginx
worker_processes auto;
error_log /var/log/nginx/error.log warn;

events {
    worker_connections 1024;
}

http {
    include /etc/nginx/mime.types;
    default_type application/octet-stream;
    
    log_format main '$remote_addr - $request - $status';
    access_log /var/log/nginx/access.log main;
    
    sendfile on;
    keepalive_timeout 65;
    
    server {
        listen 80;
        server_name ejemplo.com;
        
        location / {
            # Manejo de peticiones
        }
    }
}
"""

print(estructura)

bloques = {
    "global": "Configuración del proceso (workers, logs de error)",
    "events": "Manejo de conexiones de red",
    "http": "Toda la configuración web/HTTP",
    "server": "Un servidor virtual (dominio/puerto)",
    "location": "Reglas para URLs específicas",
}

for bloque, descripcion in bloques.items():
    print(f"  {bloque}: {descripcion}")

## Reverse Proxy Básico a Docker

> La directiva proxy_pass envía la petición al contenedor Docker que corre en localhost. El contenedor escucha en un puerto interno (ej: 8000) que no está expuesto a internet directamente, pero nginx sí puede alcanzarlo.

In [ ]:
config_proxy = """
server {
    listen 80;
    server_name mi-api.com www.mi-api.com;
    
    location / {
        proxy_pass http://localhost:8000;
        
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        
        proxy_connect_timeout 60s;
        proxy_read_timeout 60s;
        proxy_send_timeout 60s;
        
        proxy_buffering on;
        proxy_buffer_size 4k;
        proxy_buffers 8 4k;
    }
    
    location /health {
        return 200 'OK';
        add_header Content-Type text/plain;
    }
}
"""

print(config_proxy)

docker_compose = """
services:
  api:
    image: mi-fastapi-app
    ports:
      - "8000:8000"
    restart: unless-stopped
"""

print("docker-compose.yml:")
print(docker_compose)

## Múltiples Apps en el Mismo Servidor

> Nginx puede servir múltiples aplicaciones usando diferentes bloques server con distintos server_name. Esto permite tener api.tudominio.com, app.tudominio.com y admin.tudominio.com en el mismo VPS.

In [ ]:
multi_app_config = """
# API FastAPI (puerto 8000)
server {
    listen 80;
    server_name api.ejemplo.com;
    
    location / {
        proxy_pass http://localhost:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}

# Frontend React (puerto 3000)
server {
    listen 80;
    server_name app.ejemplo.com;
    
    location / {
        proxy_pass http://localhost:3000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}

# Admin panel (puerto 8080)
server {
    listen 80;
    server_name admin.ejemplo.com;
    
    allow 203.0.113.5;
    deny all;
    
    location / {
        proxy_pass http://localhost:8080;
        proxy_set_header Host $host;
    }
}
"""

print(multi_app_config)

estructura_archivos = """
/etc/nginx/
├── nginx.conf
├── sites-available/
│   ├── api.ejemplo.com
│   ├── app.ejemplo.com
│   └── admin.ejemplo.com
└── sites-enabled/
    ├── api.ejemplo.com -> ../sites-available/
    └── app.ejemplo.com -> ../sites-available/
"""

print("Estructura de archivos nginx:")
print(estructura_archivos)

## Servir Archivos Estáticos (React Build)

> Para un frontend React/Vite, lo más eficiente es que nginx sirva directamente los archivos estáticos del build. Es mucho más rápido que pasar por Node.js o un servidor de desarrollo.

In [ ]:
config_react = """
server {
    listen 80;
    server_name mi-app.com www.mi-app.com;
    
    root /var/www/mi-app/dist;
    index index.html;
    
    location / {
        try_files $uri $uri/ /index.html;
    }
    
    location /assets/ {
        expires 1y;
        add_header Cache-Control "public, immutable";
        access_log off;
    }
    
    location = /favicon.ico {
        expires 30d;
        access_log off;
    }
    
    location /api/ {
        proxy_pass http://localhost:8000/;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
    }
}
"""

print(config_react)

deploy_frontend = """
# npm run build
# scp -r dist/ deploy@tu-servidor:/var/www/mi-app/
# rsync -avz --delete dist/ deploy@tu-servidor:/var/www/mi-app/dist/
"""

print("Comandos para desplegar:")
print(deploy_frontend)

## Gzip y Headers de Caché

> La compresión gzip reduce el tamaño de las respuestas hasta un 70-80%, mejorando la velocidad. Los headers de caché evitan que el navegador descargue recursos que no han cambiado.

In [ ]:
config_performance = """
http {
    gzip on;
    gzip_vary on;
    gzip_proxied any;
    gzip_comp_level 6;
    gzip_min_length 1000;
    gzip_types
        text/plain text/css text/javascript
        application/javascript application/json
        application/xml image/svg+xml;
    
    server {
        add_header X-Frame-Options "SAMEORIGIN";
        add_header X-Content-Type-Options "nosniff";
        add_header X-XSS-Protection "1; mode=block";
        
        location ~* \.(js|css|png|jpg|jpeg|gif|ico|woff2)$ {
            expires 365d;
            add_header Cache-Control "public, immutable";
        }
        
        location ~* \.html$ {
            expires -1;
            add_header Cache-Control "no-cache, no-store, must-revalidate";
        }
    }
}
"""

print(config_performance)

verificacion = """
# curl -H "Accept-Encoding: gzip" -I https://tu-app.com/
# Content-Encoding: gzip  <- confirma que está comprimiendo
"""

print("Verificar gzip:")
print(verificacion)

## Rate Limiting con limit_req_zone

> El rate limiting limita cuántas peticiones puede hacer una IP por segundo. Es esencial para proteger la API de abusos, ataques DDoS y bots. Nginx lo implementa de forma eficiente con muy poca memoria.

In [ ]:
config_rate_limit = """
http {
    limit_req_zone $binary_remote_addr zone=api:10m rate=10r/s;
    limit_req_zone $binary_remote_addr zone=login:10m rate=1r/s;
    limit_req_zone $binary_remote_addr zone=public:10m rate=30r/s;
    
    server {
        location /api/ {
            limit_req zone=api burst=20 nodelay;
            limit_req_status 429;
            proxy_pass http://localhost:8000/api/;
        }
        
        location /api/auth/login {
            limit_req zone=login burst=5 nodelay;
            limit_req_status 429;
            proxy_pass http://localhost:8000/api/auth/login;
        }
    }
}
"""

print(config_rate_limit)

respuesta_limite = {
    "status": 429,
    "mensaje": "Too Many Requests",
    "retry_after": "1 segundo",
}

print("\nRespuesta cuando se supera el rate limit:")
for clave, valor in respuesta_limite.items():
    print(f"  {clave}: {valor}")

## Configuración Completa de Producción

nginx.conf completo para una aplicación con API FastAPI y frontend React en el mismo servidor.

In [ ]:
def generar_config_nginx(dominio, puerto_api=8000, puerto_frontend=None):
    config_completa = f"""
worker_processes auto;
error_log /var/log/nginx/error.log warn;
pid /var/run/nginx.pid;

events {{
    worker_connections 1024;
}}

http {{
    include /etc/nginx/mime.types;
    default_type application/octet-stream;
    
    sendfile on;
    keepalive_timeout 65;
    server_tokens off;
    
    limit_req_zone $binary_remote_addr zone=api:10m rate=20r/s;
    limit_req_zone $binary_remote_addr zone=login:10m rate=2r/s;
    
    gzip on;
    gzip_comp_level 6;
    gzip_types text/plain text/css application/javascript application/json image/svg+xml;
    
    # API Server
    server {{
        listen 80;
        server_name api.{dominio};
        
        location / {{
            limit_req zone=api burst=30 nodelay;
            proxy_pass http://localhost:{puerto_api};
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_set_header X-Forwarded-Proto $scheme;
        }}
    }}
    
    # Frontend Server
    server {{
        listen 80;
        server_name {dominio} www.{dominio};
        
        root /var/www/{dominio}/dist;
        index index.html;
        
        location / {{
            try_files $uri $uri/ /index.html;
        }}
        
        location /assets/ {{
            expires 1y;
            add_header Cache-Control "public, immutable";
        }}
    }}
}}
"""
    return config_completa

config = generar_config_nginx(dominio="miempresa.com", puerto_api=8000)
print(config)

print("\nVerificar: nginx -t")
print("Recargar: systemctl reload nginx")

## Tips y Mejores Prácticas

> Siempre verifica la sintaxis con nginx -t antes de recargar. Un error de sintaxis puede tumbar nginx completamente si haces systemctl restart.

> La diferencia entre systemctl reload y systemctl restart: reload recarga la configuración sin cortar conexiones activas. Úsalo en producción para evitar interrupciones.

> Un 502 Bad Gateway significa que nginx no puede alcanzar el contenedor Docker. Verifica que el contenedor esté corriendo (docker ps) y que el puerto sea el correcto.

> Usa server_tokens off en el bloque http para que nginx no revele su versión en los headers de respuesta.

> El bloque try_files $uri $uri/ /index.html es esencial para aplicaciones de una sola página (SPA) como React.

> El rate limiting por defecto devuelve 503. Configura limit_req_status 429 para usar el código HTTP correcto ("Too Many Requests").

## Errores Comunes

### Modificar nginx.conf directamente sin hacer nginx -t primero

¿Por qué ocurre?
- Un error de sintaxis en nginx.conf hace que nginx no pueda recargar, dejando el servidor sin funcionar.

Solución
- Siempre edita, luego ejecuta nginx -t para verificar sintaxis, y solo si pasa el test ejecuta systemctl reload nginx.

### Olvidar los headers proxy_set_header en el reverse proxy

¿Por qué ocurre?
- Sin estos headers, tu aplicación verá la IP de nginx (127.0.0.1) en lugar de la IP real del cliente.

Solución
- Agrega siempre: proxy_set_header X-Real-IP $remote_addr y proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for.

### No configurar try_files para aplicaciones React/Vue/Angular

¿Por qué ocurre?
- Al refrescar la página en una ruta como /dashboard, nginx busca el archivo físico y devuelve 404.

Solución
- Agrega: location / { try_files $uri $uri/ /index.html; }

### Configurar rate limiting pero olvidar el parámetro burst

¿Por qué ocurre?
- Sin burst, incluso un pequeño pico de tráfico legítimo activará el rate limit.

Solución
- Usa limit_req zone=api burst=20 nodelay; El burst permite ráfagas cortas sin penalizar al usuario.

### No configurar cache headers para archivos estáticos

¿Por qué ocurre?
- El navegador descargará todos los JS, CSS e imágenes en cada visita, haciendo la app lenta.

Solución
- Agrega expires 1y y Cache-Control: public, immutable para archivos con hash en el nombre.